# Apply the multi-timescale transfer-operator pipeline to your own data

This notebook walks through the pipeline of Kaur, Jain, Berman (PRX Life,
2026) on a generic $d$-dimensional time series.  The steps mirror the
manuscript's three-system analyses (Lorenz, *C. elegans*, *Drosophila*) and
each cell is annotated with the corresponding figure / equation.

**You provide:**

- `X` — a NumPy array of shape `(T, d)` (rows are time points; columns are
  measurement channels, e.g. joint angles, eigenworms, neural-population
  PCs);
- `fs` — the sampling frequency in Hz;
- a frequency range `(fmin, fmax)` for the wavelet decomposition (rule of
  thumb: $f_\mathrm{min}$ above DC drift, $f_\mathrm{max}$ below
  $f_\mathrm{Nyquist}/2$);
- a working lag `tau` in seconds (chosen later by the implied-timescales
  plateau).

The notebook returns:

- delay-embedded representation `X_embedded`;
- cluster sequence `states`;
- transition matrix `T`, stationary distribution `pi`, leading eigenvalues
  and eigenvectors;
- spectral-gap-selected $M$ and the G-PCCA soft basin memberships `chi`;
- arm-and-hub geometry in the leading eigenvectors;
- diagnostic plots: Cao $E_1(d)$, $\Delta H(N)$, $\tau$-plateau, eigenvalue
  spectrum, eigenspace 3D, $\chi_j(t)$.

**Expected runtime.** Depends on `T`, `d`, `N`.  Rule of thumb on a 2024
laptop:

- wavelets: $\approx 1$ s per $10^5$ frames per channel;
- $k$-means at $N = 250$: $\approx 1$ min per $10^6$ frames;
- G-PCCA at $M = 4$: $\approx 30$ s on $N = 1000$.


In [ ]:
# --- Colab / local setup ---
# If you're running on Colab, uncomment the next two lines to install the
# pip dependencies that aren't in the default Colab image (and to clone
# this repo for the helper modules + data).
#
# !pip install pygpcca powerlaw umap-learn
# !git clone https://github.com/<your-org>/slowmode.git
# %cd slowmode

import os, sys, pickle, time
import numpy as np
import matplotlib.pyplot as plt

# Ensure slowmode/ is on the path.
sys.path.insert(0, os.path.abspath('.'))

import pipeline as pp
import gpcca_utils as gu
import figures as fg

fg.setup_style()
os.makedirs('outputs', exist_ok=True)


## 0. Provide your data

Replace the example below with your own `(T, d)` array.  We use the
chaotic Lorenz attractor (without modulation) as a stand-in so the rest of
the notebook runs end-to-end.


In [ ]:
# === USER INPUT ===
import lorenz_simulation as ls
t, xyz, _ = ls.simulate(beta=0.0, T=600.0, discard=50.0, fs=100.0, seed=0)
X = xyz                    # shape (T, d) — your time series here
fs = 100.0                 # sampling frequency (Hz)

# Wavelet parameters (tune to your signal):
fmin, fmax = 0.5, 25.0
n_freqs = 25

# Pipeline parameters (we will tune these below):
d_embed = 7                # delay embedding; finalize via Cao
N = 200                    # cluster count; finalize via Delta H(N)
tau_seconds = 0.5          # working lag; finalize via tau-plateau
M = 2                      # number of basins; finalize via spectral gap

print(f'X shape = {X.shape}; sampling at {fs} Hz')


## 1. Wavelet decomposition

Morlet wavelet amplitudes with $\omega_0 = 5$ across `n_freqs` dyadically
spaced bands in $[f_\mathrm{min}, f_\mathrm{max}]$.  Output is
`(T, d * n_freqs)` (channel-major).


In [ ]:
amps, freqs = pp.morlet_wavelet_amplitudes(X, fs=fs, fmin=fmin, fmax=fmax,
                                            n_freqs=n_freqs)
print(f'wavelet amplitudes shape = {amps.shape}')
print(f'freq range = [{freqs[0]:.2f}, {freqs[-1]:.2f}] Hz')


## 2. PCA with phase-shuffle threshold

We keep the components whose eigenvalues exceed the mean of $\lambda_1$
across phase-shuffled copies of the wavelet matrix.  Increase `n_shuffles`
for tighter estimates.


In [ ]:
proj, n_kept, eigvals_pca, thresh = pp.pca_with_shuffle_threshold(
    amps, n_shuffles=10, max_keep=20, seed=0)
print(f'kept {n_kept} components above threshold {thresh:.3g}')

fig, ax = plt.subplots(figsize=(4.0, 2.8))
ax.plot(np.arange(1, len(eigvals_pca) + 1), eigvals_pca, 'o-')
ax.axhline(thresh, ls='--', color='r', label='shuffle threshold')
ax.set_yscale('log')
ax.set_xlabel('PC index'); ax.set_ylabel('eigenvalue'); ax.legend()
plt.tight_layout(); plt.show()


## 3. Cao's $E_1(d)$ to choose the embedding dimension

We pick the smallest $d$ at which $E_1(d)$ saturates near 1.  In the
manuscript this is $d = 7$ for Lorenz/worms and $d = 12$ for flies.


In [ ]:
E1 = pp.cao_e1(proj[:, :n_kept], max_d=15, tau=1, n_samples=20000, seed=0)
fig, ax = plt.subplots(figsize=(4.0, 2.8))
ax.plot(np.arange(2, len(E1) + 2), E1, 'o-')
ax.axhline(1.0, ls=':', color='0.6')
ax.set_xlabel('$d$'); ax.set_ylabel('$E_1(d)$')
plt.tight_layout(); plt.show()

# Set d_embed to the chosen saturation point (e.g. by eye on the plot).
print(f'current choice: d_embed = {d_embed}')


## 4. Delay embedding


In [ ]:
X_embedded = pp.delay_embed(proj[:, :n_kept], d=d_embed, tau=1)
print(f'delay-embedded shape = {X_embedded.shape}')


## 5. Choose $N$ via the entropy gap $\Delta H(N)$

$\Delta H(N) = H_\mathrm{shuf}(N) - H(N)$ is largest at the optimal $N$
(Methods, Sec. "Choosing the number of clusters").


In [ ]:
N_values = [50, 100, 200, 400, 800]
lag_demo = int(tau_seconds * fs)
Ns, H, Hs = pp.entropy_gap(X_embedded, N_values, lag=lag_demo,
                            framerate=fs, seed=0, n_init=5, verbose=True)
fig, ax = plt.subplots(figsize=(4.0, 2.8))
ax.plot(Ns, Hs - H, 'o-')
ax.set_xscale('log'); ax.set_xlabel('$N$')
ax.set_ylabel(r'$\Delta H(N)$ (bits/s)')
plt.tight_layout(); plt.show()

# Adjust N to the maximum of Delta H(N) above (or stick with the demo N).
print(f'current choice: N = {N}')


## 6. K-means partition


In [ ]:
states = pp.kmeans_partition(X_embedded, N=N, n_init=20, seed=0)
print(f'partitioned into N = {N} clusters; sequence length = {len(states):,}')


## 7. Choose $\tau$ via the implied-timescales plateau

Sweep $\tau$ and look for a range over which the leading implied
timescales $t_k = -\tau / \log|\lambda_k(\tau)|$ are roughly constant
in $\tau$.  Pick the working lag inside that plateau.


In [ ]:
lags_s = np.array([0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0])
implied = []
for ls_ in lags_s:
    lg = max(int(ls_ * fs), 1)
    Tt = pp.make_transition_matrix(states, lag=lg, n_states=N)
    e, _ = pp.leading_eigvecs(Tt, k=4)
    implied.append(-(lg / fs) / np.log(np.maximum(np.abs(e[:4]), 1e-12)))
implied = np.array(implied)
fig, ax = plt.subplots(figsize=(4.4, 3.0))
for k in range(3):
    ax.plot(lags_s, implied[:, k], 'o-', label=fr'$t_{{{k+2}}}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\tau$ (s)'); ax.set_ylabel('implied timescale (s)')
ax.legend()
plt.tight_layout(); plt.show()

print(f'current choice: tau = {tau_seconds} s')


## 8. Transition matrix and eigendecomposition


In [ ]:
lag = int(tau_seconds * fs)
T = pp.make_transition_matrix(states, lag=lag, n_states=N)
pi = pp.stationary_distribution(T)
evals, evecs = pp.leading_eigvecs(T, k=10)
print(f'leading |lambda_k| = {np.abs(evals)[:6].round(4)}')

# Spectral gap selection of M.
M_pick, gap, ratios = gu.select_M_spectral_gap(evals, M_min=2, M_max=6)
print(f'spectral gap selects M = {M_pick} (ratio = {gap:.3f})')


## 9. G-PCCA at the chosen $M$


In [ ]:
out = gu.run_gpcca(T, M=M, eta=pi)
chi = out['chi']
print(f'crispness    = {out["crispness"]:.3f}')
print(f'basin counts = {out["basin_counts"]}')
print(f'pi per basin = {out["pi_basin"].round(3)}')


## 10. Eigenvalue spectrum


In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 2.8))
fg.plot_eigenvalue_dots(ax, evals[:10], M=M)
plt.tight_layout(); plt.show()


## 11. 3D eigenspace with hub and arms


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa
phi = evecs[:, :3].real
geo = gu.compute_hub_arms(phi, pi, chi)

fig = plt.figure(figsize=(4.6, 4.0))
ax = fig.add_subplot(111, projection='3d')
fg.plot_eigenspace_3d(ax, phi, pi=pi, chi=chi,
                       hub=geo['hub'], arm_dirs=geo['arm_dirs'],
                       arm_centroids=geo['arm_centroids'])
plt.tight_layout(); plt.show()


## 12. $\chi_j(t)$ time series

Soft basin memberships over time, smoothed with a 5 s rolling window.


In [ ]:
chi_traces = chi[states]
T_show = min(len(chi_traces), 30 * 60 * int(fs))   # up to 30 minutes
t_show = np.arange(T_show) / fs

fig, ax = plt.subplots(figsize=(7.2, 2.6))
fg.plot_chi_timeseries(ax, t_show, chi_traces[:T_show],
                        smoothing_window=int(5 * fs))
plt.tight_layout(); plt.show()


## 13. Optional: $r_k(\tau)$ non-Markovianity signature

If your dynamics are Markovian at the chosen state resolution,
$r_k(\tau) = -\log|\lambda_k(\tau)|/\tau$ is constant in $\tau$.  The
strong decrease of $r_2$ with $\tau$ is the operator-level non-Markovianity
documented in Berman et al. (2016) and reproduced in Fig. 5B of the
manuscript.


In [ ]:
lags_s = np.logspace(-1, 1, 12)
rk = []
for ls_ in lags_s:
    lg = max(int(ls_ * fs), 1)
    Tt = pp.make_transition_matrix(states, lag=lg, n_states=N)
    e, _ = pp.leading_eigvecs(Tt, k=4)
    rk.append(-np.log(np.maximum(np.abs(e[:4]), 1e-12)) / (lg / fs))
rk = np.array(rk)
fig, ax = plt.subplots(figsize=(4.4, 3.0))
for k in range(3):
    ax.plot(lags_s, rk[:, k], 'o-', label=fr'$r_{{{k+2}}}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\tau$ (s)'); ax.set_ylabel(r'$r_k(\tau)$ (1/s)')
ax.legend()
plt.tight_layout(); plt.show()


---

## Recap

You've now run the full multi-timescale transfer-operator pipeline on your
own data, with the same diagnostic criteria the paper uses.  All cached
quantities (`X_embedded`, `states`, `T`, `pi`, `chi`, `geo`) are available
in your kernel for downstream analyses (basin-level Markov modeling,
$r_k(\tau)$ over a wider grid, dwell-time fits, behavior-map enrichment
maps, leave-one-individual-out CV, etc.).  See `flies.ipynb` for a worked
example of the full set.


## 14. Optional: ground-truth correlation (Lorenz Fig. 2D recipe)

If you have an external "hidden driver" $h(t)$ aligned to your time series
(synthetic data, or a known modulator), `pipeline.phi2_correlation_preet`
computes $|r(\phi_2(t), h(t))|$ using the per-seed characteristic-timescale
lag and Savitzky-Golay smoothing convention used in the manuscript Fig. 2D
$\beta$ sweep.

```python
r, lag, t_c = pp.phi2_correlation_preet(states, h, fs=fs)
print(f'|r| = {r:.3f}    lag = {lag}    t_c = {t_c:.1f} frames')
```

The recipe is: $T(\mathrm{lag}=1)$ → $t_c = -1/\log|\lambda_2|$ →
$T(\mathrm{round}(t_c))$ → leading non-trivial right eigenvector by
descending real-eigenvalue → savgol-smooth → Pearson.
